# Imports and Configuration
- **`INPUT_PATH`**: folder containing the input data to process.
- **`OUTPUT_PATH`**: folder where results will be written.

In [ ]:
import os
import numpy as np
import tifffile
from scipy import ndimage

INPUT_PATH = r"../data/images"
OUTPUT_PATH = r"../outputs/sharpness"

# If the output folder does not exist, create it automatically.
# os.makedirs(OUTPUT_PATH, exist_ok=True)

print(f"Input folder : {INPUT_PATH}")
print(f"Output folder: {OUTPUT_PATH}")

# Metrics Definition

Define the image-quality metric used in this notebook.

- **`tenengrad_sharpness(image, mask=None, threshold=0.0)`**: Tenengrad sharpness score. The Sobel gradients are computed with `scipy.ndimage.sobel`, and the score is the mean squared gradient magnitude, optionally restricted to a `mask`. Higher values indicate a sharper image.

In [ ]:
def tenengrad_sharpness(image, mask=None, threshold=0.0):
    """Tenengrad sharpness score (mean squared Sobel gradient magnitude).

    Uses `scipy.ndimage.sobel` to compute the gradients.  Only gradient
    magnitudes above `threshold` are kept, and the score is the mean of those
    squared magnitudes.  If `mask` is given, the score is computed only over
    the masked pixels.  A higher score means a sharper image.
    """
    img = np.asarray(image, dtype=np.float64)

    gx = ndimage.sobel(img, axis=1, mode="nearest")
    gy = ndimage.sobel(img, axis=0, mode="nearest")
    g_squared = gx ** 2 + gy ** 2

    # Tenengrad keeps only gradients above the threshold
    g_squared[g_squared <= threshold ** 2] = 0.0

    values = g_squared[mask] if mask is not None else g_squared
    if values.size == 0:
        return None

    return float(np.mean(values))

In [ ]:
def load_probability(prob_path, prob_channel=0):
    """Load an ilastik probability TIFF and return the foreground-probability map (2D).

    Handles (H, W), (H, W, C), and (C, H, W) layouts.  Float values stay as-is;
    integer probability maps are rescaled to [0, 1].
    """
    prob_all = tifffile.imread(prob_path)
    print(prob_all.shape)
    prob = prob_all[:,:,prob_channel]

    # Not needed if prob is already between 0 to 1 .
    if np.issubdtype(prob_all.dtype, np.integer):
        prob = prob / float(np.iinfo(prob_all.dtype).max)

    return prob

In [ ]:
def sharpness_from_probability(raw_path, prob_path, threshold=0.5, fg_channel=0):
    """Load raw + foreground-probability map, threshold, and compute Tenengrad sharpness."""
    raw = tifffile.imread(raw_path)
    fg_prob = load_probability(prob_path, prob_channel=fg_channel)
    if raw.shape != fg_prob.shape:
        raise ValueError(f"Shape mismatch: raw {raw.shape} vs prob {fg_prob.shape}")

    fg_mask = fg_prob >= threshold
    return tenengrad_sharpness(raw, mask=fg_mask)

# File Discovery

Function that walks `INPUT_PATH` and lists `.tif` files, optionally pairing each one with a matching companion file.

- **`find_files(root, token=None)`**:
  - If `token` is `None`: returns every `.tif` file under `root`, paired with an empty string.
  - If `token` is e.g. `"_Probabilities"` or `"_Simple Segmentation"`: only returns raws that have a matching `<raw>_<token>.tif` companion.

Returns a list of `(raw_path, companion_path)` tuples.

In [ ]:
def find_files(root, token=None):
    """Walk `root` and return list of (raw_path, companion_path) tuples.

    If `token` is None, all .tif files are returned with an empty companion path.
    Otherwise, only raws that have a matching `<raw>_<token>.tif` companion are kept.
    """
    pairs = []
    for dirpath, _, filenames in os.walk(root):
        for fname in filenames:
            if not fname.lower().endswith(".tif"):
                continue
            full = os.path.join(dirpath, fname)

            if token is None:
                pairs.append((full, ""))
                continue

            # skip companion files themselves; only treat plain raws as candidates
            if token in fname:
                continue

            stem = fname.rsplit(".", 1)[0]
            companion = os.path.join(dirpath, stem + token + ".tif")
            if os.path.exists(companion):
                pairs.append((full, companion))

    return sorted(pairs)

## Sharpness from Segmentation masks

In [ ]:
FG_LABEL = 1

seg_pairs = find_files(INPUT_PATH, token="_Simple Segmentation")
print(f"Found {len(seg_pairs)} segmentation pair(s) under {INPUT_PATH}")
print(f"Foreground label = {FG_LABEL}\n")

for raw_path, seg_path in seg_pairs:
    raw = tifffile.imread(raw_path)
    seg = tifffile.imread(seg_path)

    fg_mask = (seg == FG_LABEL)
    sharpness = tenengrad_sharpness(raw, mask=fg_mask)

    print("--------------")
    print(raw_path)
    print("Sharpness (Tenengrad)", sharpness)
    print("--------------")

# Sharpness from Probability Maps

Use ilastik foreground-probability TIFFs to define the foreground mask via a threshold, then compute the Tenengrad sharpness on the raw image.

Reuses `load_probability` and `sharpness_from_probability` defined earlier.

In [ ]:
THRESHOLD = 0.5
FG_CHANNEL = 0

prob_pairs = find_files(INPUT_PATH, token="_Probabilities")
print(f"Found {len(prob_pairs)} probability pair(s) under {INPUT_PATH}")

for raw_path, prob_path in prob_pairs:
    sharpness = sharpness_from_probability(raw_path, prob_path, threshold=THRESHOLD, fg_channel=FG_CHANNEL)
    print("--------------")
    print(raw_path)
    print("Sharpness (Tenengrad)", sharpness)
    print("--------------")